# Research 52 — Loyalty/platform and engagement/device blend screen

Evaluate the two independently trained seed-42 interaction models with predeclared fixed 80/20 mixtures against the current reference. Show standalone performance, leader context, macro and weighted F1, and fold deltas. No fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert (sums > 0).all(), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
p_leader = load_probs("segment-vip-region-four-seed-average-oof.csv")
p_loyalty_platform = load_probs("segment-lightgbm-loyalty-platform-oof.csv")
p_engagement_device = load_probs("segment-lightgbm-engagement-device-oof.csv")
p_loyalty_blend = 0.8 * p_base + 0.2 * p_loyalty_platform
p_engagement_blend = 0.8 * p_base + 0.2 * p_engagement_device
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
candidates = {
    "Current reference": p_base,
    "Four-seed VIP × region leader": p_leader,
    "Loyalty × platform standalone": p_loyalty_platform,
    "Loyalty × platform, fixed 80/20": p_loyalty_blend,
    "Engagement × device standalone": p_engagement_device,
    "Engagement × device, fixed 80/20": p_engagement_blend,
}
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    if name not in ["Current reference", "Four-seed VIP × region leader"]:
        for fold, (_, valid_idx) in enumerate(folds, start=1):
            score = f1_score(y[valid_idx], pred[valid_idx], average="macro")
            baseline = f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro")
            row[f"fold_{fold}_macro_f1"] = score
            row[f"fold_{fold}_delta_vs_reference"] = score - baseline
    rows.append(row)
results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-loyalty-platform-engagement-device-screen.csv", index=False)
for probs, filename in [
    (p_loyalty_blend, "segment-lightgbm-loyalty-platform-blend-oof.csv"),
    (p_engagement_blend, "segment-lightgbm-engagement-device-blend-oof.csv"),
]:
    pd.DataFrame(probs, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
        ROOT / "predictions" / filename, index=False
    )
print("Fixed candidate-only 80/20 blends evaluated; no weights searched; Test not loaded.")

,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_1_delta_vs_reference,fold_2_macro_f1,fold_2_delta_vs_reference,fold_3_macro_f1,fold_3_delta_vs_reference
0,Current reference,0.750895,0.764683,NaN,NaN,NaN,NaN,NaN,NaN
1,Four-seed VIP × region leader,0.752172,0.765704,NaN,NaN,NaN,NaN,NaN,NaN
2,Loyalty × platform standalone,0.741465,0.753623,0.740502,-0.008943,0.739146,-0.010675,0.744728,-0.008661
3,"Loyalty × platform, fixed 80/20",0.751441,0.764929,0.750020,0.000575,0.750160,0.000339,0.754118,0.000728
4,Engagement × device standalone,0.741810,0.754052,0.740359,-0.009086,0.739762,-0.010059,0.745292,-0.008097
5,"Engagement × device, fixed 80/20",0.751298,0.764844,0.749748,0.000303,0.750630,0.000810,0.753488,0.000099


Fixed candidate-only 80/20 blends evaluated; no weights searched; Test not loaded.
